In [3]:
# ============================================
# ПОЛНЫЙ КОД ДЛЯ RTX 3080 (10GB VRAM)
# ============================================

import os
import cv2
import gc
import copy
import random
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedKFold
from torch.optim.lr_scheduler import OneCycleLR

import warnings
warnings.filterwarnings('ignore')

# ============================================
# 1. КОНФИГУРАЦИЯ ДЛЯ RTX 3080 (10GB)
# ============================================
class Config:
    # ПУТЬ К ДАННЫМ - ИЗМЕНИТЕ НА ВАШ!
    data_dir = Path('D:/DL_4')  # <--- ПРОВЕРЬТЕ ЭТОТ ПУТЬ!
    
    train_csv_path = data_dir / "train.csv"
    train_img_dir = data_dir / "train" / "train"  # если внутри ещё папка train
    test_img_dir = data_dir / "test" / "test"
    
    # Уменьшенные размеры для RTX 3080
    img_height = 512
    img_width = 192
    
    backbone = 'tf_efficientnet_b0.ns_jft_in1k'
    pretrained = True
    dropout = 0.3
    
    vit_dim = 384
    vit_heads = 6
    vit_layers = 8
    vit_mlp_ratio = 4
    
    # Оптимизированные параметры для 10GB VRAM
    n_folds = 2  # всего 2 фолда (хватит для хорошего результата)
    batch_size = 32  # запас по памяти
    num_workers = 0  # на Windows лучше 0
    epochs = 50
    learning_rate = 5e-4
    weight_decay = 0.01
    
    # Progressive Resizing - без 768x288 (не влезет в память)
    progressive_epochs = [0, 20, 40]
    progressive_sizes = [(256, 96), (384, 144), (512, 192)]
    
    use_amp = True
    gradient_clip = 1.0
    early_stopping_patience = 15
    
    label_smoothing = 0.05
    focal_gamma = 2.0
    num_chars = 10

config = Config()

# Исправляем пути, если внутри папок нет вложенных папок train/test
if not (config.train_img_dir).exists():
    # пробуем без вложенной папки
    config.train_img_dir = config.data_dir / "train"
    config.test_img_dir = config.data_dir / "test"
    print(f"📁 Используем пути: train={config.train_img_dir}, test={config.test_img_dir}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

print("="*50)
print(f"🚀 GPU: {torch.cuda.get_device_name(0)}")
print(f"📦 Batch size: {config.batch_size}")
print(f"📏 Финальный размер: {config.img_height}x{config.img_width}")
print(f"📁 Данные: {config.data_dir}")
print("="*50)

# ============================================
# 2. ПРЕПРОЦЕССИНГ
# ============================================
def estimate_noise_level(image_gray):
    diff_x = np.abs(np.diff(image_gray, axis=1))[:, :-1]
    diff_y = np.abs(np.diff(image_gray, axis=0))[:-1, :]
    return (np.std(diff_x) + np.std(diff_y)) / 2

def adaptive_denoise(image_rgb):
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    noise_level = estimate_noise_level(gray)
    
    lab = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    
    clip_limit = 2.0 if noise_level < 30 else 3.0
    tile_size = 8 if noise_level > 40 else 6
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile_size, tile_size))
    l = clahe.apply(l)
    
    if noise_level > 30:
        l = cv2.bilateralFilter(l, 5, 50, 50)
    
    lab = cv2.merge([l, a, b])
    image_denoised = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    
    if noise_level > 20:
        image_denoised = cv2.convertScaleAbs(image_denoised, alpha=1.15, beta=5)
    
    return image_denoised

def preprocess_ocr_image(image, target_size=None):
    if len(image.shape) == 3 and image.shape[2] == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    if target_size is None:
        target_size = (config.img_width, config.img_height)
    
    image = cv2.resize(image, target_size)
    image = adaptive_denoise(image)
    
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.equalizeHist(l)
    lab = cv2.merge([l, a, b])
    image = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    
    return image

# ============================================
# 3. АУГМЕНТАЦИИ
# ============================================
train_augmentations = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.6),
    A.RandomGamma(gamma_limit=(70, 130), p=0.4),
    A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=25, val_shift_limit=15, p=0.4),
    A.GaussNoise(var_limit=(10.0, 40.0), p=0.5),
    A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=0.5),
    A.Sharpen(alpha=(0.2, 0.6), lightness=(0.5, 1.0), p=0.4),
    A.CoarseDropout(max_holes=3, max_height=20, max_width=40, p=0.25),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_augmentations = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# ============================================
# 4. DATASET
# ============================================
class OCRDataset(Dataset):
    def __init__(self, filenames, labels, augmentations=None, base_path=config.train_img_dir,
                 is_test=False, epoch=0):
        self.filenames = filenames
        self.labels = labels
        self.augmentations = augmentations
        self.base_path = base_path
        self.is_test = is_test
        self.epoch = epoch
        self.update_size(epoch)
    
    def update_size(self, epoch):
        for start_epoch, (h, w) in zip(config.progressive_epochs, config.progressive_sizes):
            if epoch >= start_epoch:
                self.img_height = h
                self.img_width = w
            else:
                break
    
    def __len__(self):
        return len(self.filenames)
    
    def __getitem__(self, idx):
        img_path = os.path.join(self.base_path, self.filenames[idx])
        image = cv2.imread(img_path)
        
        if image is None:
            image = np.zeros((self.img_height, self.img_width, 3), dtype=np.uint8)
        
        image = preprocess_ocr_image(image, target_size=(self.img_width, self.img_height))
        
        if self.augmentations:
            augmented = self.augmentations(image=image)
            image = augmented['image']
        
        if self.is_test:
            return image, self.filenames[idx]
        else:
            label = torch.tensor([int(d) + 1 for d in str(self.labels[idx])], dtype=torch.long)
            return image, label

def collate_fn(batch):
    images = torch.stack([item[0] for item in batch])
    targets = [item[1] for item in batch]
    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    
    max_len = max(target_lengths)
    padded_targets = torch.zeros(len(targets), max_len, dtype=torch.long)
    for i, t in enumerate(targets):
        padded_targets[i, :len(t)] = t
    
    return images, padded_targets, target_lengths

# ============================================
# 5. МОДЕЛЬ
# ============================================
class ViTForOCR(nn.Module):
    def __init__(self, num_chars=10, img_height=512, img_width=192,
                 embed_dim=384, num_heads=6, num_layers=8, mlp_ratio=4, dropout=0.3):
        super().__init__()
        
        self.backbone = timm.create_model(
            config.backbone,
            pretrained=config.pretrained,
            features_only=True,
            out_indices=[4]
        )
        
        with torch.no_grad():
            dummy = torch.randn(1, 3, img_height, img_width)
            feats = self.backbone(dummy)
            cnn_channels = feats[-1].shape[1]
            cnn_height = feats[-1].shape[2]
            cnn_width = feats[-1].shape[3]
        
        print(f"📊 Backbone: {config.backbone}")
        print(f"📊 Патчей: {cnn_height}x{cnn_width} = {cnn_height * cnn_width}")
        
        self.cnn_proj = nn.Sequential(
            nn.Conv2d(cnn_channels, embed_dim, kernel_size=1),
            nn.BatchNorm2d(embed_dim),
            nn.GELU(),
            nn.Conv2d(embed_dim, embed_dim, kernel_size=3, padding=1, groups=embed_dim),
            nn.BatchNorm2d(embed_dim),
            nn.GELU(),
        )
        
        self.num_patches = cnn_height * cnn_width
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches, embed_dim) * 0.02)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=int(embed_dim * mlp_ratio), dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(embed_dim)
        
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, embed_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(embed_dim, embed_dim // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, num_chars + 1)
        )
    
    def forward(self, x):
        feats = self.backbone(x)
        x = feats[-1]
        x = self.cnn_proj(x)
        _, _, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)
        x = x + self.pos_embed[:, :H*W, :]
        x = self.transformer(x)
        x = self.norm(x)
        return self.classifier(x)

class FocalCTCLoss(nn.Module):
    def __init__(self, blank=0, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.blank = blank
        self.gamma = gamma
        self.label_smoothing = label_smoothing
    
    def forward(self, logits, targets, target_lengths):
        input_lengths = torch.full((logits.size(0),), logits.size(1), dtype=torch.long, device=logits.device)
        
        if self.label_smoothing > 0:
            logits = logits * (1 - self.label_smoothing) + self.label_smoothing / logits.size(-1)
        
        log_probs = F.log_softmax(logits, dim=2)
        probs = log_probs.exp()
        focal_weight = (1 - probs) ** self.gamma
        log_probs_focal = (log_probs * focal_weight).permute(1, 0, 2)
        
        return F.ctc_loss(log_probs_focal, targets, input_lengths, target_lengths,
                          blank=self.blank, zero_infinity=True)

# ============================================
# 6. УТИЛИТЫ
# ============================================
def decode_predictions(logits):
    preds = logits.argmax(dim=2)
    decoded = []
    for pred in preds:
        result = []
        prev = 0
        for p in pred.cpu().numpy():
            if p != 0 and p != prev:
                result.append(str(p - 1))
            prev = p
        decoded.append(''.join(result))
    return decoded

def compute_accuracy(preds, targets):
    if len(preds) == 0:
        return 0.0
    return sum(1 for p, t in zip(preds, targets) if p == t) / len(preds)

def postprocess_prediction(pred):
    if not pred:
        return pred
    pred = pred.lstrip('0')
    if not pred:
        pred = '0'
    if len(pred) > 4:
        pred = pred[:4]
    return pred

class EarlyStopping:
    def __init__(self, patience=15, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, score):
        if self.best_score is None:
            self.best_score = score
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.counter = 0
        return self.early_stop

# ИСПРАВЛЕННЫЙ SWA
class SWA:
    def __init__(self, model, start_epoch):
        self.model = model
        self.start_epoch = start_epoch
        self.swa_model = copy.deepcopy(model)
        self.n_models = 0

    def update(self, epoch):
        if epoch >= self.start_epoch:
            self.n_models += 1
            for swa_param, model_param in zip(self.swa_model.parameters(), self.model.parameters()):
                swa_param.data = swa_param.data * (self.n_models - 1) / self.n_models + model_param.data / self.n_models

    def get_model(self):
        return self.swa_model

# ============================================
# 7. ФУНКЦИИ ОБУЧЕНИЯ
# ============================================
def train_epoch(model, loader, optimizer, criterion, scaler, device, scheduler=None, epoch=0):
    model.train()
    total_loss = 0
    all_preds = []
    all_targets = []
    
    if hasattr(loader.dataset, 'update_size'):
        loader.dataset.update_size(epoch)
    
    pbar = tqdm(loader, desc=f"Train E{epoch}")
    for images, targets, target_lengths in pbar:
        images = images.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)
        
        optimizer.zero_grad(set_to_none=True)
        
        with autocast(enabled=config.use_amp):
            logits = model(images)
            loss = criterion(logits, targets, target_lengths)
        
        scaler.scale(loss).backward()
        if config.gradient_clip:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)
        scaler.step(optimizer)
        scaler.update()
        
        if scheduler:
            scheduler.step()
        
        total_loss += loss.item()
        
        preds = decode_predictions(logits)
        targets_str = [''.join(str(int(x)-1) for x in t[:l].cpu().numpy()) 
                      for t, l in zip(targets, target_lengths)]
        all_preds.extend(preds)
        all_targets.extend(targets_str)
        
        batch_acc = compute_accuracy(preds, targets_str)
        pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{batch_acc:.4f}")
    
    return total_loss/len(loader), compute_accuracy(all_preds, all_targets), 0

@torch.no_grad()
def validate(model, loader, criterion, device, epoch=0):
    model.eval()
    total_loss = 0
    all_preds = []
    all_targets = []
    
    pbar = tqdm(loader, desc=f"Valid E{epoch}")
    for images, targets, target_lengths in pbar:
        images = images.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)
        
        with autocast(enabled=config.use_amp):
            logits = model(images)
            loss = criterion(logits, targets, target_lengths)
        
        total_loss += loss.item()
        
        preds = decode_predictions(logits)
        targets_str = [''.join(str(int(x)-1) for x in t[:l].cpu().numpy()) 
                      for t, l in zip(targets, target_lengths)]
        all_preds.extend(preds)
        all_targets.extend(targets_str)
        
        batch_acc = compute_accuracy(preds, targets_str)
        pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{batch_acc:.4f}")
    
    return total_loss/len(loader), compute_accuracy(all_preds, all_targets), 0

# ============================================
# 8. ЗАГРУЗКА ДАННЫХ
# ============================================
print("\n📥 Загрузка данных...")
train_df = pd.read_csv(config.train_csv_path)
train_df['price_len'] = train_df['Price'].astype(str).str.len()
train_df['digits_str'] = train_df['Price'].astype(str)
train_df['stratum'] = train_df['price_len']
print(f"📊 Датасет: {len(train_df)} строк")

# ============================================
# 9. ОБУЧЕНИЕ
# ============================================
device = torch.device("cuda")
skf = StratifiedKFold(n_splits=config.n_folds, shuffle=True, random_state=SEED)

fold_results = []
best_model_paths = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['stratum'])):
    print(f"\n{'='*50}")
    print(f"🔄 FOLD {fold + 1}/{config.n_folds}")
    print(f"{'='*50}")
    
    train_dataset = OCRDataset(
        train_df.iloc[train_idx]['Filename'].values,
        train_df.iloc[train_idx]['digits_str'].values,
        augmentations=train_augmentations, is_test=False, epoch=0
    )
    val_dataset = OCRDataset(
        train_df.iloc[val_idx]['Filename'].values,
        train_df.iloc[val_idx]['digits_str'].values,
        augmentations=val_augmentations, is_test=False, epoch=0
    )
    
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True,
                              num_workers=config.num_workers, collate_fn=collate_fn, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size * 2, shuffle=False,
                            num_workers=config.num_workers, collate_fn=collate_fn)
    
    model = ViTForOCR(
        num_chars=config.num_chars, img_height=config.img_height, img_width=config.img_width,
        embed_dim=config.vit_dim, num_heads=config.vit_heads, num_layers=config.vit_layers,
        dropout=config.dropout
    ).to(device)
    
    criterion = FocalCTCLoss(blank=0, gamma=config.focal_gamma, label_smoothing=config.label_smoothing)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    
    total_steps = len(train_loader) * config.epochs
    scheduler = OneCycleLR(optimizer, max_lr=config.learning_rate, total_steps=total_steps,
                           pct_start=0.2, div_factor=25, final_div_factor=1000)
    
    scaler = GradScaler(enabled=config.use_amp)
    early_stopping = EarlyStopping(patience=config.early_stopping_patience)
    swa = SWA(model, config.epochs - 10)  # SWA на последних 10 эпохах
    
    best_val_acc = 0
    best_model_path = f'best_model_fold_{fold+1}.pth'
    
    for epoch in range(1, config.epochs + 1):
        print(f"\n📚 Epoch {epoch}/{config.epochs}")
        
        current_size = config.progressive_sizes[-1]
        for start_epoch, size in zip(config.progressive_epochs, config.progressive_sizes):
            if epoch >= start_epoch:
                current_size = size
        
        print(f"📐 Размер: {current_size[0]}x{current_size[1]}")
        print(f"📈 LR: {optimizer.param_groups[0]['lr']:.2e}")
        
        train_dataset.update_size(epoch)
        val_dataset.update_size(epoch)
        
        train_loss, train_acc, _ = train_epoch(model, train_loader, optimizer, criterion,
                                               scaler, device, scheduler, epoch)
        val_loss, val_acc, _ = validate(model, val_loader, criterion, device, epoch)
        
        print(f"📊 Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")
        
        swa.update(epoch)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'model_state_dict': model.state_dict(),
                'val_acc': val_acc,
                'fold': fold + 1,
                'epoch': epoch
            }, best_model_path)
            print(f"✅ Сохранено! Val Acc: {val_acc:.4f}")
        
        if early_stopping(val_acc):
            print(f"🛑 Early stopping на эпохе {epoch}")
            break
        
        torch.cuda.empty_cache()
        gc.collect()
    
    # SWA финальная оценка
    swa_model = swa.get_model()
    _, swa_acc, _ = validate(swa_model, val_loader, criterion, device, epoch=config.epochs)
    print(f"🏆 SWA Val Acc: {swa_acc:.4f}")
    
    if swa_acc > best_val_acc:
        best_val_acc = swa_acc
        torch.save({
            'model_state_dict': swa_model.state_dict(),
            'val_acc': swa_acc,
            'fold': fold + 1,
            'epoch': config.epochs,
            'is_swa': True
        }, best_model_path)
        print(f"✅ Сохранена SWA модель с acc: {swa_acc:.4f}")
    
    fold_results.append(best_val_acc)
    best_model_paths.append(best_model_path)
    print(f"\n🏆 Fold {fold+1} лучшая точность: {best_val_acc:.4f}")
    
    del model, swa_model
    torch.cuda.empty_cache()
    gc.collect()

print(f"\n{'='*50}")
print(f"📊 РЕЗУЛЬТАТЫ КРОСС-ВАЛИДАЦИИ")
print(f"{'='*50}")
print(f"Точность по фолдам: {[f'{x:.4f}' for x in fold_results]}")
print(f"Средняя точность: {np.mean(fold_results):.4f} (+/- {np.std(fold_results):.4f})")

# ============================================
# 10. ИНФЕРЕНС
# ============================================
print("\n🔮 Создание submission.csv...")

submission_path = config.data_dir / "sample_submission.csv"
submission = pd.read_csv(submission_path)
print(f"📊 Тестовых изображений: {len(submission)}")

# Загружаем лучшую модель
best_fold = np.argmax(fold_results)
best_model_path = best_model_paths[best_fold]
print(f"📁 Используем модель: {best_model_path} (acc={fold_results[best_fold]:.4f})")

checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

predictions = []
for idx, row in tqdm(submission.iterrows(), total=len(submission)):
    img_path = config.test_img_dir / row['Filename']
    image = cv2.imread(str(img_path))
    
    if image is None:
        predictions.append("")
        continue
    
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    img_processed = preprocess_ocr_image(image, target_size=(config.img_width, config.img_height))
    img_tensor = val_augmentations(image=img_processed)['image'].unsqueeze(0).to(device)
    
    with torch.no_grad():
        logits = model(img_tensor)
        pred = decode_predictions(logits)[0]
        predictions.append(postprocess_prediction(pred))

submission["Price"] = predictions
submission.to_csv("submission.csv", index=False)

print("\n✅ submission.csv создан!")
print(f"📁 Сохранён как: {os.path.abspath('submission.csv')}")
print("\n📋 ПЕРВЫЕ 10 ПРЕДСКАЗАНИЙ:")
print(submission.head(10))

# ============================================
# 11. ИТОГИ
# ============================================
print(f"\n{'='*50}")
print(f"🎉 ГОТОВО!")
print(f"{'='*50}")
print(f"📁 Файлы в текущей папке:")
for f in os.listdir('.'):
    if f.endswith('.pth') or f == 'submission.csv':
        size = os.path.getsize(f) / (1024*1024)
        print(f"   {f} ({size:.1f} MB)")
print(f"\n💡 submission.csv можно отправлять на Kaggle")

🚀 GPU: NVIDIA GeForce RTX 3080
📦 Batch size: 32
📏 Финальный размер: 512x192
📁 Данные: D:\DL_4

📥 Загрузка данных...
📊 Датасет: 15050 строк

🔄 FOLD 1/2


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


📊 Backbone: tf_efficientnet_b0.ns_jft_in1k
📊 Патчей: 16x6 = 96

📚 Epoch 1/50
📐 Размер: 256x96
📈 LR: 2.00e-05


Valid E1: 100%|██████████| 118/118 [00:14<00:00,  8.07it/s, acc=0.0000, loss=1.9804]


📊 Train Acc: 0.0001, Val Acc: 0.0000

📚 Epoch 2/50
📐 Размер: 256x96
📈 LR: 3.18e-05


...

Valid E50: 100%|██████████| 118/118 [00:37<00:00,  3.11it/s, acc=1.0000, loss=-0.0044]


📊 Train Acc: 0.9922, Val Acc: 0.9814


Valid E50: 100%|██████████| 118/118 [00:33<00:00,  3.52it/s, acc=0.0000, loss=36.2252]


🏆 SWA Val Acc: 0.0000

🏆 Fold 2 лучшая точность: 0.9823

📊 РЕЗУЛЬТАТЫ КРОСС-ВАЛИДАЦИИ
Точность по фолдам: ['0.9849', '0.9823']
Средняя точность: 0.9836 (+/- 0.0013)

🔮 Создание submission.csv...
📊 Тестовых изображений: 3762
📁 Используем модель: best_model_fold_1.pth (acc=0.9849)


NameError: name 'model' is not defined

In [4]:
# ============================================
# СОЗДАНИЕ SUBMISSION ФАЙЛОВ (3 варианта)
# ============================================
import os
import torch
import cv2
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from collections import Counter

# Пути
data_dir = Path('D:/DL_4')
test_img_dir = data_dir / "test" / "test"
submission_path = data_dir / "sample_submission.csv"

# Проверяем существование папки с тестами
if not test_img_dir.exists():
    test_img_dir = data_dir / "test"  # если без вложенной папки

print("🔍 ПРОВЕРКА ДАННЫХ")
print("="*40)
print(f"📁 Тестовые изображения: {test_img_dir}")
print(f"   Существует: {test_img_dir.exists()}")
if test_img_dir.exists():
    test_files = list(test_img_dir.glob("*.jpg"))
    print(f"   Найдено файлов: {len(test_files)}")

# Загружаем sample_submission
if not submission_path.exists():
    print(f"❌ {submission_path} не найден!")
    # Пробуем альтернативные пути
    alt_paths = [
        data_dir / "sample_submission.csv",
        Path("./sample_submission.csv"),
        Path("../sample_submission.csv")
    ]
    for p in alt_paths:
        if p.exists():
            submission_path = p
            print(f"✅ Найден: {p}")
            break

if not submission_path.exists():
    print("❌ Нужно скачать sample_submission.csv")
    # Создаём заглушку
    print("📝 Создаём временный файл...")
    !kaggle competitions download -c dl-lab-4-ocr -p . 2>/dev/null
    import zipfile
    if os.path.exists("dl-lab-4-ocr.zip"):
        with zipfile.ZipFile("dl-lab-4-ocr.zip", 'r') as zf:
            if "sample_submission.csv" in zf.namelist():
                zf.extract("sample_submission.csv")
                submission_path = Path("sample_submission.csv")

submission = pd.read_csv(submission_path)
print(f"📊 Тестовых изображений: {len(submission)}")

# Находим все сохранённые модели
model_files = []
for f in os.listdir('.'):
    if f.startswith('best_model_fold_') and f.endswith('.pth'):
        model_files.append(f)
        print(f"✅ Найдена модель: {f}")

if not model_files:
    print("❌ Нет сохранённых моделей!")
    # Ищем в других местах
    for f in os.listdir('./'):
        if f.endswith('.pth'):
            model_files.append(f)
            print(f"✅ Альтернативная модель: {f}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"💻 Device: {device}")

# Загружаем все модели
models = []
model_accs = []

for model_path in model_files:
    print(f"\n🎭 Загрузка {model_path}...")
    
    # Создаём новую модель для каждого файла
    model = ViTForOCR(
        num_chars=config.num_chars,
        img_height=config.img_height,
        img_width=config.img_width,
        embed_dim=config.vit_dim,
        num_heads=config.vit_heads,
        num_layers=config.vit_layers,
        dropout=config.dropout
    ).to(device)
    
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    models.append(model)
    
    acc = checkpoint.get('val_acc', 0)
    model_accs.append(acc)
    print(f"   ✅ Загружена (val_acc: {acc:.4f})")

print(f"\n🎯 Загружено моделей: {len(models)}")

# ============================================
# ФУНКЦИЯ ДЛЯ ИНФЕРЕНСА
# ============================================
def predict_with_model(model, image_path):
    image = cv2.imread(str(image_path))
    if image is None:
        return ""
    
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    img_processed = preprocess_ocr_image(image, target_size=(config.img_width, config.img_height))
    img_tensor = val_augmentations(image=img_processed)['image'].unsqueeze(0).to(device)
    
    with torch.no_grad():
        logits = model(img_tensor)
        pred = decode_predictions(logits)[0]
        return postprocess_prediction(pred)

# ============================================
# 1. СОЗДАЁМ SUBMISSION ДЛЯ КАЖДОГО ФОЛДА
# ============================================
print("\n" + "="*50)
print("📁 СОЗДАНИЕ SUBMISSION ДЛЯ КАЖДОЙ МОДЕЛИ")
print("="*50)

submissions = []

for idx, (model, model_path) in enumerate(zip(models, model_files)):
    print(f"\n🔮 Модель {idx+1}: {model_path}")
    predictions = []
    
    for _, row in tqdm(submission.iterrows(), total=len(submission), desc=f"Model {idx+1}"):
        img_path = test_img_dir / row['Filename']
        pred = predict_with_model(model, img_path)
        predictions.append(pred)
    
    # Сохраняем
    sub = submission.copy()
    sub["Price"] = predictions
    sub_name = f"submission_fold_{idx+1}.csv"
    sub.to_csv(sub_name, index=False)
    submissions.append(sub)
    print(f"✅ Сохранён: {sub_name}")

# ============================================
# 2. АНСАМБЛЕВЫЙ SUBMISSION (ГОЛОСОВАНИЕ)
# ============================================
print("\n" + "="*50)
print("🤝 СОЗДАНИЕ АНСАМБЛЕВОГО SUBMISSION")
print("="*50)

ensemble_predictions = []

for row_idx in tqdm(range(len(submission)), desc="Ensemble"):
    # Собираем предсказания всех моделей
    preds = []
    for sub in submissions:
        preds.append(sub.iloc[row_idx]["Price"])
    
    # Голосование (выбираем самое частое предсказание)
    final_pred = Counter(preds).most_common(1)[0][0]
    ensemble_predictions.append(final_pred)

ensemble_sub = submission.copy()
ensemble_sub["Price"] = ensemble_predictions
ensemble_sub.to_csv("submission_ensemble.csv", index=False)
print(f"✅ Сохранён: submission_ensemble.csv")

# ============================================
# 3. ВЗВЕШЕННЫЙ АНСАМБЛЬ (С УЧЁТОМ ТОЧНОСТИ МОДЕЛЕЙ)
# ============================================
print("\n" + "="*50)
print("⚖️ ВЗВЕШЕННЫЙ АНСАМБЛЬ (по точности моделей)")
print("="*50)

# Нормализуем веса
weights = np.array(model_accs)
weights = weights / weights.sum()
print(f"Веса моделей: {[f'{w:.3f}' for w in weights]}")

weighted_predictions = []

for row_idx in tqdm(range(len(submission)), desc="Weighted Ensemble"):
    # Для каждого возможного предсказания считаем взвешенную сумму
    all_preds = {}
    for sub, weight in zip(submissions, weights):
        pred = sub.iloc[row_idx]["Price"]
        if pred not in all_preds:
            all_preds[pred] = 0
        all_preds[pred] += weight
    
    # Выбираем предсказание с максимальным весом
    final_pred = max(all_preds, key=all_preds.get)
    weighted_predictions.append(final_pred)

weighted_sub = submission.copy()
weighted_sub["Price"] = weighted_predictions
weighted_sub.to_csv("submission_weighted.csv", index=False)
print(f"✅ Сохранён: submission_weighted.csv")

# ============================================
# 4. СРАВНЕНИЕ РЕЗУЛЬТАТОВ
# ============================================
print("\n" + "="*50)
print("📊 СРАВНЕНИЕ SUBMISSION ФАЙЛОВ")
print("="*50)

print(f"\n📁 Созданные файлы:")
for f in os.listdir('.'):
    if f.startswith('submission') and f.endswith('.csv'):
        size = os.path.getsize(f) / 1024
        print(f"   {f} ({size:.1f} KB)")

# Показываем пример предсказаний
print(f"\n📋 ПРИМЕР ПРЕДСКАЗАНИЙ (первые 10):")
for i in range(min(10, len(submission))):
    print(f"   {submission.iloc[i]['Filename'][:40]}... ", end="")
    for idx, sub in enumerate(submissions):
        print(f"F{idx+1}:{sub.iloc[i]['Price']} ", end="")
    print(f"| Ансамбль:{ensemble_sub.iloc[i]['Price']} | Взвеш:{weighted_sub.iloc[i]['Price']}")

# ============================================
# 5. СКАЧИВАНИЕ ВСЕХ ФАЙЛОВ
# ============================================
print("\n" + "="*50)
print("📥 СКАЧИВАНИЕ ФАЙЛОВ")
print("="*50)

# Для локального ПК — просто показываем пути
print("\n📁 Файлы сохранены в текущей папке:")
for f in os.listdir('.'):
    if f.startswith('submission') and f.endswith('.csv'):
        abs_path = os.path.abspath(f)
        print(f"   📄 {abs_path}")
    
    if f.startswith('best_model_fold_') and f.endswith('.pth'):
        abs_path = os.path.abspath(f)
        size = os.path.getsize(f) / (1024*1024)
        print(f"   🧠 {abs_path} ({size:.1f} MB)")

print("\n" + "="*50)
print("🎉 ГОТОВО!")
print("="*50)
print("\n💡 Что вы получили:")
print("   1. submission_fold_1.csv — предсказания первой модели")
print("   2. submission_fold_2.csv — предсказания второй модели")
print("   3. submission_ensemble.csv — ансамбль голосованием")
print("   4. submission_weighted.csv — ансамбль с весами моделей")
print("\n📌 Рекомендация: отправьте на Kaggle ВСЕ 4 файла")
print("   и выберите тот, который даст лучший результат!")

🔍 ПРОВЕРКА ДАННЫХ
📁 Тестовые изображения: D:\DL_4\test\test
   Существует: True
   Найдено файлов: 3762
📊 Тестовых изображений: 3762
✅ Найдена модель: best_model_fold_1.pth
✅ Найдена модель: best_model_fold_2.pth
💻 Device: cuda

🎭 Загрузка best_model_fold_1.pth...


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


📊 Backbone: tf_efficientnet_b0.ns_jft_in1k
📊 Патчей: 16x6 = 96


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


   ✅ Загружена (val_acc: 0.9849)

🎭 Загрузка best_model_fold_2.pth...
📊 Backbone: tf_efficientnet_b0.ns_jft_in1k
📊 Патчей: 16x6 = 96
   ✅ Загружена (val_acc: 0.9823)

🎯 Загружено моделей: 2

📁 СОЗДАНИЕ SUBMISSION ДЛЯ КАЖДОЙ МОДЕЛИ

🔮 Модель 1: best_model_fold_1.pth


...

✅ Сохранён: submission_fold_2.csv

🤝 СОЗДАНИЕ АНСАМБЛЕВОГО SUBMISSION


Ensemble: 100%|██████████| 3762/3762 [00:00<00:00, 34445.44it/s]


✅ Сохранён: submission_ensemble.csv

⚖️ ВЗВЕШЕННЫЙ АНСАМБЛЬ (по точности моделей)
Веса моделей: ['0.501', '0.499']


Weighted Ensemble: 100%|██████████| 3762/3762 [00:00<00:00, 28856.51it/s]

✅ Сохранён: submission_weighted.csv

📊 СРАВНЕНИЕ SUBMISSION ФАЙЛОВ

📁 Созданные файлы:
   submission_ensemble.csv (181.9 KB)
   submission_fold_1.csv (181.9 KB)
   submission_fold_2.csv (181.8 KB)
   submission_weighted.csv (181.9 KB)

📋 ПРИМЕР ПРЕДСКАЗАНИЙ (первые 10):
   8_D0-CF-13-24-59-DC_2026-01-22-14-02-47.... F1:34 F2:34 | Ансамбль:34 | Взвеш:34
   12_10-B4-1D-E0-1E-60_2026-01-22-14-00-35... F1:129 F2:129 | Ансамбль:129 | Взвеш:129
   9_D0-CF-13-22-6F-AC_2026-01-22-14-03-36.... F1:109 F2:109 | Ансамбль:109 | Взвеш:109
   15_10-B4-1D-C8-0C-D8_2026-01-22-14-02-31... F1:199 F2:199 | Ансамбль:199 | Взвеш:199
   28_D0-CF-13-24-CD-28_2026-01-22-18-28-43... F1:39 F2:59 | Ансамбль:39 | Взвеш:39
   7_D0-CF-13-23-CC-E8_2026-01-22-14-00-13.... F1:85 F2:85 | Ансамбль:85 | Взвеш:85
   22_D0-CF-13-22-59-C8_2026-01-22-22-04-31... F1:129 F2:129 | Ансамбль:129 | Взвеш:129
   0_10-B4-1D-C8-01-E0_36_2025-11-15-10-59-... F1:99 F2:99 | Ансамбль:99 | Взвеш:99
   9_D0-CF-13-23-6F-E8_2026-01-22-17-58-1